In [1]:

# Operating system paths and environment variables
import os
import json
import ast

# Import Path from pathlib
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd

# Load the saved machine learning model
import joblib

# Combine TF-IDF feature vectors
from scipy.sparse import hstack

# TF-IDF baseline retrieval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Semantic embeddings and FAISS retrieval
import faiss
from sentence_transformers import SentenceTransformer

# Display DataFrames in the notebook
from IPython.display import display


In [2]:
# Define the project directory
PROJECT_DIR = Path.cwd().parent

# Define the Task B dataset folder
task_b_path = PROJECT_DIR / "Data" / "TaskB"

# Define training and validation folders
training_path = task_b_path / "training"
validation_path = task_b_path / "validation"

# Check whether the required folders exist
if not task_b_path.exists():
    raise FileNotFoundError(f"Task B folder not found: {task_b_path}")

if not training_path.exists():
    raise FileNotFoundError(f"Training folder not found: {training_path}")

if not validation_path.exists():
    raise FileNotFoundError(f"Validation folder not found: {validation_path}")

# Find the saved model files inside the project
model_files = list(PROJECT_DIR.rglob("tuned_linear_svm_model.pkl"))
tfidf_files = list(PROJECT_DIR.rglob("tfidf_vectorizer.pkl"))

# Check whether the model files were found
if not model_files:
    raise FileNotFoundError("Saved Linear SVM model was not found.")

if not tfidf_files:
    raise FileNotFoundError("Saved TF-IDF vectorizer was not found.")

# Load the saved Linear SVM model
saved_svm_model = joblib.load(model_files[0])

# Load the saved TF-IDF vectorizer
saved_tfidf = joblib.load(tfidf_files[0])

print("Task B folder exists:", task_b_path.exists())
print("Linear SVM model loaded from:", model_files[0])
print("TF-IDF vectorizer loaded from:", tfidf_files[0])
print("Both files loaded successfully!")

Task B folder exists: True
Linear SVM model loaded from: c:\Users\jagad\Downloads\Job-Skill-Extraction-and-Job-Title-Matching using NLP\Data\Downloads\tuned_linear_svm_model.pkl
TF-IDF vectorizer loaded from: c:\Users\jagad\Downloads\Job-Skill-Extraction-and-Job-Title-Matching using NLP\Data\Downloads\tfidf_vectorizer.pkl
Both files loaded successfully!


In [3]:
# Provide two sample job titles
sample_title_1 = "Data Scientist"
sample_title_2 = "Machine Learning Engineer"


# Convert both job titles into TF-IDF features
sample_vector_title1 = saved_tfidf.transform([sample_title_1])
sample_vector_title2 = saved_tfidf.transform([sample_title_2])


# Combine the two feature vectors in the same format used by the model
sample_vector = hstack([
    sample_vector_title1,
    sample_vector_title2
])


# Predict the occupation URI
sample_prediction = saved_svm_model.predict(sample_vector)[0]


# Display the prediction
print("Job Title 1:", sample_title_1)
print("Job Title 2:", sample_title_2)
print("Predicted Occupation URI:", sample_prediction)

Job Title 1: Data Scientist
Job Title 2: Machine Learning Engineer
Predicted Occupation URI: http://data.europa.eu/esco/occupation/35553663-deab-4d9a-bf22-15c1625d28e8


In [4]:
# Load job-to-skill relationships
job2skill_file = training_path / "job2skill.tsv"

if not job2skill_file.exists():
    raise FileNotFoundError(
        f"Job-skill file not found: {job2skill_file}"
    )

job2skill_df = pd.read_csv(
    job2skill_file,
    sep="\t",
    header=None,
    names=["occupation_uri", "skill_uri", "relation"]
)

# Load occupation ID to job-title mappings
with open(
    training_path / "jobid2terms.json",
    "r",
    encoding="utf-8"
) as file:
    jobid2terms = json.load(file)

# Load skill ID to skill-name mappings
with open(
    training_path / "skillid2terms.json",
    "r",
    encoding="utf-8"
) as file:
    skillid2terms = json.load(file)

print("Job-skill data shape:", job2skill_df.shape)
print("Number of job IDs:", len(jobid2terms))
print("Number of skill IDs:", len(skillid2terms))

display(job2skill_df.head())

Job-skill data shape: (114699, 3)
Number of job IDs: 3039
Number of skill IDs: 13939


,occupation_uri,skill_uri,relation
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential


In [5]:
# Get the main readable term from a mapping
def get_main_term(terms):
    if isinstance(terms, list):
        return terms[0] if terms else None
    return terms


# Create readable occupation-title mappings
job_lookup = {
    job_id: get_main_term(terms)
    for job_id, terms in jobid2terms.items()
}

# Create readable skill-name mappings
skill_lookup = {
    skill_id: get_main_term(terms)
    for skill_id, terms in skillid2terms.items()
}

# Add job titles and skill names to the relationship DataFrame
job2skill_df["job_title"] = (
    job2skill_df["occupation_uri"].map(job_lookup)
)

job2skill_df["skill_name"] = (
    job2skill_df["skill_uri"].map(skill_lookup)
)

print("Missing job titles:", job2skill_df["job_title"].isna().sum())
print("Missing skill names:", job2skill_df["skill_name"].isna().sum())

display(job2skill_df.head())

Missing job titles: 0
Missing skill names: 0


,occupation_uri,skill_uri,relation,job_title,skill_name
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential,technical director,promote health and safety
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential,technical director,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential,technical director,negotiate health and safety issues with third ...
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential,technical director,theatre techniques
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential,technical director,coordinate technical teams in artistic product...


In [6]:
# Separate essential and optional skills
essential_skills_df = job2skill_df[
    job2skill_df["relation"] == "essential"
].copy()

optional_skills_df = job2skill_df[
    job2skill_df["relation"] == "optional"
].copy()

# Group essential skills by occupation
essential_grouped = (
    essential_skills_df
    .groupby("occupation_uri")["skill_name"]
    .apply(lambda skills: skills.dropna().tolist())
    .reset_index(name="essential_skills")
)

# Group optional skills by occupation
optional_grouped = (
    optional_skills_df
    .groupby("occupation_uri")["skill_name"]
    .apply(lambda skills: skills.dropna().tolist())
    .reset_index(name="optional_skills")
)

# Combine both skill groups
job_skills_df = pd.merge(
    essential_grouped,
    optional_grouped,
    on="occupation_uri",
    how="outer"
)

# Replace missing skill lists with empty lists
for column in ["essential_skills", "optional_skills"]:
    job_skills_df[column] = job_skills_df[column].apply(
        lambda value: value if isinstance(value, list) else []
    )

# Add readable job titles
job_skills_df["job_title"] = (
    job_skills_df["occupation_uri"].map(job_lookup)
)

print("Combined dataset shape:", job_skills_df.shape)
print("Unique occupations:", job_skills_df["occupation_uri"].nunique())

display(job_skills_df.head())

Combined dataset shape: (3011, 4)
Unique occupations: 3011


,occupation_uri,essential_skills,optional_skills,job_title
0,http://data.europa.eu/esco/occupation/00030d09...,"[promote health and safety, organise rehearsal...",[write technical riders],technical director
1,http://data.europa.eu/esco/occupation/000e93a3...,"[monitor moving workpiece in a machine, ensure...","[metal drawing processes, tend wire drawing ma...",metal drawing machine operator
2,http://data.europa.eu/esco/occupation/0019b951...,"[read standard blueprints, meet deadlines, qua...","[write records for repairs, operate precision ...",precision device inspector
3,http://data.europa.eu/esco/occupation/0022f466...,"[common aviation safety regulations, assist in...","[perform aircraft maintenance, test electrical...",air traffic safety technician
4,http://data.europa.eu/esco/occupation/002da35b...,"[property management software, manage hospital...","[develop working procedures, manage budgets, m...",hospitality revenue manager


In [7]:
# Create a lookup for occupation-specific skills
job_skill_lookup = job_skills_df.set_index(
    "occupation_uri"
).to_dict(orient="index")

# Find skills for the predicted occupation
skill_result = job_skill_lookup.get(sample_prediction)

if skill_result is None:
    print("Predicted occupation not found in Task B.")
    print("Predicted URI:", sample_prediction)

else:
    print("Predicted Job Title:", skill_result["job_title"])

    print("\nESSENTIAL SKILLS")
    for number, skill in enumerate(
        skill_result["essential_skills"], start=1
    ):
        print(f"{number}. {skill}")

    print("\nOPTIONAL SKILLS")
    for number, skill in enumerate(
        skill_result["optional_skills"], start=1
    ):
        print(f"{number}. {skill}")

    print("\nSUMMARY")
    print("Essential skills:", len(skill_result["essential_skills"]))
    print("Optional skills:", len(skill_result["optional_skills"]))
    print(
        "Total skills:",
        len(skill_result["essential_skills"])
        + len(skill_result["optional_skills"])
    )

Predicted Job Title:  artificial intelligence engineer 

ESSENTIAL SKILLS
1. develop statistical software
2. unstructured data
3. analyse big data
4. principles of artificial intelligence
5. algorithms
6. artificial neural networks
7. apply ICT systems theory
8. develop creative ideas
9. information categorisation
10. digital data processing
11. information architecture
12. Python (computer programming)
13. computer programming
14. business process modelling
15. task algorithmisation
16. design process
17. information structure
18. computer simulation
19. create data sets
20. systems development life-cycle
21. use data processing techniques
22. analyse business requirements
23. data science
24. define technical requirements
25. creatively use digital technologies
26. visual presentation techniques
27. information extraction
28. resource description framework query language
29. deliver visual presentation of data
30. data models
31. data mining

OPTIONAL SKILLS
1. Haskell
2. Common Lisp

In [8]:
# Load validation job-title queries
queries_df = pd.read_csv(
    validation_path / "queries",
    sep="\t"
)

# Load candidate skills and their aliases
corpus_df = pd.read_csv(
    validation_path / "corpus_elements",
    sep="\t"
)

# Load query-skill relevance judgments
qrels_df = pd.read_csv(
    validation_path / "qrels.tsv",
    sep="\t",
    header=None,
    names=["query_id", "iteration", "corpus_id", "relevance"]
)

# Display dataset sizes
print("Queries shape:", queries_df.shape)
print("Candidate skills shape:", corpus_df.shape)
print("Relevance judgments shape:", qrels_df.shape)

# Display columns to verify the file formats
print("\nQuery columns:", queries_df.columns.tolist())
print("Candidate skill columns:", corpus_df.columns.tolist())
print("Qrels columns:", qrels_df.columns.tolist())

# Preview each dataset
display(queries_df.head())
display(corpus_df.head())
display(qrels_df.head())

Queries shape: (304, 2)
Candidate skills shape: (1439, 3)
Relevance judgments shape: (25888, 4)

Query columns: ['q_id', 'jobtitle']
Candidate skill columns: ['c_id', 'esco_uri', 'skill_aliases']
Qrels columns: ['query_id', 'iteration', 'corpus_id', 'relevance']


,q_id,jobtitle
0,dev_qb_jt_1,corporate governance analyst
1,dev_qb_jt_2,logistics business analyst
2,dev_qb_jt_3,operations planning analyst
3,dev_qb_jt_4,real estate data analyst
4,dev_qb_jt_5,security analyst ii


,c_id,esco_uri,skill_aliases
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"['pricing plans', 'price strategies', 'pricing..."
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"['putting out fires', 'put out fires', 'exting..."
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"['online assessment', 'analysis of web strateg..."
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"['keeping up with trends', 'keep pace with tre..."
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"['prepare tax returns form', 'complete tax ret..."


,query_id,iteration,corpus_id,relevance
0,dev_qb_jt_263,0,dev_cb_sk_1009,1
1,dev_qb_jt_263,0,dev_cb_sk_1014,1
2,dev_qb_jt_263,0,dev_cb_sk_1027,1
3,dev_qb_jt_263,0,dev_cb_sk_1028,1
4,dev_qb_jt_263,0,dev_cb_sk_1069,1


In [9]:
# Check the relevance labels
print("Relevance label counts:")
print(qrels_df["relevance"].value_counts().sort_index())

# Confirm required columns exist
assert {"q_id", "jobtitle"}.issubset(queries_df.columns)
assert {"c_id", "skill_aliases"}.issubset(corpus_df.columns)
assert {
    "query_id", "corpus_id", "relevance"
}.issubset(qrels_df.columns)

# Convert IDs to strings for consistent comparisons
query_ids = set(queries_df["q_id"].dropna().astype(str))
qrels_query_ids = set(qrels_df["query_id"].dropna().astype(str))

corpus_ids = set(corpus_df["c_id"].dropna().astype(str))
qrels_corpus_ids = set(qrels_df["corpus_id"].dropna().astype(str))

# Identify missing IDs
missing_queries = qrels_query_ids - query_ids
missing_skills = qrels_corpus_ids - corpus_ids

print("\nUnique query IDs:", len(query_ids))
print("Unique query IDs in qrels:", len(qrels_query_ids))
print("Query IDs missing from queries:", len(missing_queries))

print("\nUnique candidate skill IDs:", len(corpus_ids))
print("Candidate IDs in qrels:", len(qrels_corpus_ids))
print("Candidate skill IDs missing from corpus:", len(missing_skills))

# Check missing values
print("\nMissing job titles:", queries_df["jobtitle"].isna().sum())
print("Missing candidate aliases:", corpus_df["skill_aliases"].isna().sum())

Relevance label counts:
relevance
1    25888
Name: count, dtype: int64

Unique query IDs: 304
Unique query IDs in qrels: 304
Query IDs missing from queries: 0

Unique candidate skill IDs: 1439
Candidate IDs in qrels: 1365
Candidate skill IDs missing from corpus: 0

Missing job titles: 0
Missing candidate aliases: 0


In [10]:
# Create a separate copy of the candidate-skill corpus
skills_df = corpus_df.copy()

# Convert skill aliases into Python lists
def convert_aliases(aliases):

    if pd.isna(aliases):
        return []

    if isinstance(aliases, str):
        try:
            aliases = ast.literal_eval(aliases)
        except (ValueError, SyntaxError):
            aliases = [aliases]

    if isinstance(aliases, list):
        return [
            str(alias).strip()
            for alias in aliases
            if str(alias).strip()
        ]

    return [str(aliases).strip()]


# Process skill aliases
skills_df["skill_aliases"] = (
    skills_df["skill_aliases"].apply(convert_aliases)
)

# Combine aliases into a single text description
skills_df["skill_text"] = skills_df["skill_aliases"].apply(
    lambda aliases: " ".join(aliases)
)

# Remove empty skill descriptions
skills_df = skills_df[
    skills_df["skill_text"].str.strip().ne("")
].reset_index(drop=True)

# Create the TF-IDF vectorizer for baseline retrieval
skill_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    stop_words="english"
)

# Convert candidate skill descriptions into TF-IDF vectors
skill_tfidf_matrix = skill_vectorizer.fit_transform(
    skills_df["skill_text"]
)

print("Number of candidate skills:", len(skills_df))
print("TF-IDF matrix shape:", skill_tfidf_matrix.shape)
print("Missing skill texts:", skills_df["skill_text"].isna().sum())

display(skills_df[["c_id", "skill_text"]].head())

Number of candidate skills: 1439
TF-IDF matrix shape: (1439, 21985)
Missing skill texts: 0


,c_id,skill_text
0,dev_cb_sk_1,pricing plans price strategies pricing tactics...
1,dev_cb_sk_2,putting out fires put out fires extinguish fir...
2,dev_cb_sk_3,online assessment analysis of web strategy web...
3,dev_cb_sk_4,keeping up with trends keep pace with trends f...
4,dev_cb_sk_5,prepare tax returns form complete tax returns ...


In [11]:
# Retrieve candidate skills using TF-IDF similarity
def retrieve_skills(job_title, top_k=10):

    # Convert the job title into a TF-IDF vector
    query_vector = skill_vectorizer.transform([str(job_title)])

    # Calculate similarity against all candidate skills
    similarity_scores = cosine_similarity(
        query_vector,
        skill_tfidf_matrix
    ).flatten()

    # Rank candidate skills by similarity
    top_indices = np.argsort(similarity_scores)[::-1][:top_k]

    # Create the ranked results
    results = skills_df.iloc[top_indices][
        ["c_id", "esco_uri", "skill_text"]
    ].copy()

    results["similarity_score"] = similarity_scores[top_indices]

    return results.reset_index(drop=True)


# Test the TF-IDF retrieval baseline
sample_job_title = "Data Scientist"

retrieved_skills = retrieve_skills(
    sample_job_title,
    top_k=10
)

display(retrieved_skills)

,c_id,esco_uri,skill_text,similarity_score
0,dev_cb_sk_443,http://data.europa.eu/esco/skill/0d6714c6-720e...,gather data gathering data data gavering conve...,0.702261
1,dev_cb_sk_54,http://data.europa.eu/esco/skill/f2d57f41-43b4...,data processing process data processing of dat...,0.687564
2,dev_cb_sk_1173,http://data.europa.eu/esco/skill/9ff9db9d-d14b...,manage data data administration manage data li...,0.605255
3,dev_cb_sk_498,http://data.europa.eu/esco/skill/2b92a5b2-6758...,perform data analysis perform analysis of data...,0.506829
4,dev_cb_sk_742,http://data.europa.eu/esco/skill/97bd1c21-66b2...,data analytics data anlysis,0.488447
5,dev_cb_sk_514,http://data.europa.eu/esco/skill/c3e36d05-8ae8...,deliver visual presentation of data visualise ...,0.468974
6,dev_cb_sk_81,http://data.europa.eu/esco/skill/9d0d89be-bffa...,Data Migrator Elixir Repertoire for Data ETL T...,0.443081
7,dev_cb_sk_965,http://data.europa.eu/esco/skill/fecf8a0d-62c4...,data models,0.426745
8,dev_cb_sk_372,http://data.europa.eu/esco/skill/edebd83d-35f6...,science of data data science data science process,0.425803
9,dev_cb_sk_303,http://data.europa.eu/esco/skill/713fb616-118e...,verify data implement data quality processes,0.417617


In [12]:
# Keep only relevance judgments marked as relevant
positive_qrels = qrels_df[
    qrels_df["relevance"] > 0
].copy()

# Create a query-to-relevant-skills mapping
relevant_skills = (
    positive_qrels
    .groupby("query_id")["corpus_id"]
    .apply(lambda values: set(values.astype(str)))
    .to_dict()
)

# Create a query-ID-to-job-title mapping
query_titles = (
    queries_df
    .dropna(subset=["jobtitle"])
    .set_index("q_id")["jobtitle"]
    .to_dict()
)


# Evaluate TF-IDF retrieval
def evaluate_retrieval(top_k=10):

    precisions = []
    recalls = []
    hit_rates = []

    for query_id, job_title in query_titles.items():

        # Retrieve the top candidate skills
        results = retrieve_skills(
            job_title,
            top_k=top_k
        )

        # Normalize candidate IDs before comparison
        retrieved_ids = set(results["c_id"].astype(str))

        # Look up known relevant skills
        relevant_ids = relevant_skills.get(
            str(query_id),
            set()
        )

        # Skip queries without positive judgments
        if not relevant_ids:
            continue

        # Find relevant skills retrieved by the baseline
        matched_ids = retrieved_ids.intersection(relevant_ids)

        # Calculate metrics
        precision = len(matched_ids) / top_k
        recall = len(matched_ids) / len(relevant_ids)
        hit_rate = int(len(matched_ids) > 0)

        precisions.append(precision)
        recalls.append(recall)
        hit_rates.append(hit_rate)

    # Return consistent metric columns
    return {
        "K": top_k,
        "Precision": np.mean(precisions) if precisions else 0,
        "Recall": np.mean(recalls) if recalls else 0,
        "Hit Rate": np.mean(hit_rates) if hit_rates else 0,
        "Evaluated Queries": len(recalls)
    }


# Evaluate at K = 5, 10, and 20
evaluation_df = pd.DataFrame([
    evaluate_retrieval(top_k=5),
    evaluate_retrieval(top_k=10),
    evaluate_retrieval(top_k=20)
])

display(evaluation_df)

,K,Precision,Recall,Hit Rate,Evaluated Queries
0,5,0.365789,0.024264,0.812500,304
1,10,0.287171,0.036951,0.851974,304
2,20,0.218421,0.054037,0.911184,304


In [13]:
# Load a pre-trained Sentence Transformer model
# It converts text into numerical vectors called embeddings.

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Sentence Transformer loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Transformer loaded successfully!


In [14]:
# Make sure all skill descriptions are strings
skills_df["skill_text"] = (
    skills_df["skill_text"]
    .fillna("")
    .astype(str)
)

# Remove empty skill descriptions
skills_df = (
    skills_df[skills_df["skill_text"].str.strip() != ""]
    .reset_index(drop=True)
)

# Generate embeddings for every skill description
skill_embeddings = embedding_model.encode(
    skills_df["skill_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Inspect the generated embeddings
print("Number of skill embeddings:", skill_embeddings.shape[0])
print("Embedding dimensions:", skill_embeddings.shape[1])
print("Embedding data type:", skill_embeddings.dtype)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Number of skill embeddings: 1439
Embedding dimensions: 384
Embedding data type: float32


In [15]:
# Save the numerical embeddings for later use
np.save("skill_embeddings.npy", skill_embeddings)

# Save the skill records in the same row order as the embeddings
skills_df.to_pickle("skills_corpus.pkl")

print("Skill embeddings and corpus saved successfully!")

Skill embeddings and corpus saved successfully!


In [16]:
# Create a FAISS index for similarity search.
# Since the embeddings are normalized, inner product represents
# cosine similarity.

embedding_dimension = skill_embeddings.shape[1]

skill_index = faiss.IndexFlatIP(embedding_dimension)

# Add the skill embeddings to the index
skill_index.add(skill_embeddings)

print("FAISS index created successfully!")
print("Number of indexed skills:", skill_index.ntotal)
print("Embedding dimensions:", skill_index.d)

FAISS index created successfully!
Number of indexed skills: 1439
Embedding dimensions: 384


In [17]:
# Test retrieval using a sample job description
query = """
Develop machine learning models, analyze data,
build predictive systems, and evaluate model performance.
"""

# Convert the query into a normalized embedding
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Retrieve the 10 most similar skills
scores, indices = skill_index.search(query_embedding, 10)

# Get the matching skill records
results = skills_df.iloc[indices[0]][
    ["c_id", "esco_uri", "skill_text"]
].copy()

# Add the similarity scores
results["similarity_score"] = scores[0]

display(results.reset_index(drop=True))

,c_id,esco_uri,skill_text,similarity_score
0,dev_cb_sk_1382,http://data.europa.eu/esco/skill/e2887d71-8ff4...,developing models develop a model launch model...,0.627057
1,dev_cb_sk_729,http://data.europa.eu/esco/skill/49812c2d-3a36...,develop classification system formulate classi...,0.508734
2,dev_cb_sk_965,http://data.europa.eu/esco/skill/fecf8a0d-62c4...,data models,0.481936
3,dev_cb_sk_335,http://data.europa.eu/esco/skill/3f86173d-e101...,assess system performance perform system analy...,0.465407
4,dev_cb_sk_1351,http://data.europa.eu/esco/skill/6a322874-e32f...,develop automated software tests design automa...,0.449400
5,dev_cb_sk_1401,http://data.europa.eu/esco/skill/2c5cbddf-3393...,plan future capacity requirements project futu...,0.445923
6,dev_cb_sk_498,http://data.europa.eu/esco/skill/2b92a5b2-6758...,perform data analysis perform analysis of data...,0.426973
7,dev_cb_sk_843,http://data.europa.eu/esco/skill/ed39edd3-44d0...,ICT performance analysis methods performance a...,0.417363
8,dev_cb_sk_43,http://data.europa.eu/esco/skill/b16200f4-4f39...,Open source model,0.416533
9,dev_cb_sk_920,http://data.europa.eu/esco/skill/6eaeea3a-f100...,develop recommendations formulate findings for...,0.406696


In [18]:
# Save the index so it can be reused later
faiss.write_index(skill_index, "skill_faiss.index")

print("FAISS index saved successfully!")

FAISS index saved successfully!


In [19]:
# Load validation relevance judgments
qrels_path = validation_path / "qrels.tsv"

qrels_df = pd.read_csv(
    qrels_path,
    sep="\t",
    header=None,
    names=["q_id", "query_field", "c_id", "relevance"]
)

# Normalize IDs so comparisons work consistently
qrels_df["q_id"] = qrels_df["q_id"].astype(str)
qrels_df["c_id"] = qrels_df["c_id"].astype(str)

queries_df["q_id"] = queries_df["q_id"].astype(str)
skills_df["c_id"] = skills_df["c_id"].astype(str)

# Convert relevance labels to numeric values where possible
qrels_df["relevance"] = pd.to_numeric(
    qrels_df["relevance"],
    errors="coerce"
)

print("Qrels shape:", qrels_df.shape)
print("\nRelevance label distribution:")
print(qrels_df["relevance"].value_counts(dropna=False).sort_index())

display(qrels_df.head())

Qrels shape: (25888, 4)

Relevance label distribution:
relevance
1    25888
Name: count, dtype: int64


,q_id,query_field,c_id,relevance
0,dev_qb_jt_263,0,dev_cb_sk_1009,1
1,dev_qb_jt_263,0,dev_cb_sk_1014,1
2,dev_qb_jt_263,0,dev_cb_sk_1027,1
3,dev_qb_jt_263,0,dev_cb_sk_1028,1
4,dev_qb_jt_263,0,dev_cb_sk_1069,1


In [20]:
# Keep judgments with positive relevance.
# This assumes that positive numeric values indicate relevant skills.
relevant_qrels = qrels_df[
    qrels_df["relevance"] > 0
].copy()

# Build a dictionary:
# query ID -> set of relevant skill IDs
qrels_relevant = (
    relevant_qrels.groupby("q_id")["c_id"]
    .apply(set)
    .to_dict()
)

# Keep only queries that have at least one relevant skill
valid_queries = queries_df[
    queries_df["q_id"].isin(qrels_relevant.keys())
].copy()

print("Total validation queries:", len(queries_df))
print("Queries with relevant judgments:", len(valid_queries))
print("Queries excluded from evaluation:", len(queries_df) - len(valid_queries))
print("Total queries with relevant skill sets:", len(qrels_relevant))

Total validation queries: 304
Queries with relevant judgments: 304
Queries excluded from evaluation: 0
Total queries with relevant skill sets: 304


In [21]:
# Store evaluation results
evaluation_results = []

# Evaluate the top 5, 10, and 20 retrieved skills
for _, query_row in valid_queries.iterrows():

    q_id = str(query_row["q_id"])
    query_text = str(query_row["jobtitle"])

    # Convert the job title into a normalized embedding
    query_embedding = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve the top 20 skills
    scores, indices = skill_index.search(query_embedding, 20)

    # Convert the retrieved rows into skill IDs
    retrieved_ids = (
        skills_df.iloc[indices[0]]["c_id"]
        .astype(str)
        .tolist()
    )

    # Ground-truth relevant skill IDs for this query
    relevant_ids = qrels_relevant.get(q_id, set())

    # Skip queries without relevant skills
    if not relevant_ids:
        continue

    for k in [5, 10, 20]:

        top_ids = retrieved_ids[:k]

        # Count unique relevant skills retrieved
        relevant_count = len(set(top_ids) & relevant_ids)

        evaluation_results.append({
            "q_id": q_id,
            "K": k,
            "Precision": relevant_count / k,
            "Recall": relevant_count / len(relevant_ids),
            "HitRate": int(relevant_count > 0)
        })

# Convert the results into a DataFrame
evaluation_df = pd.DataFrame(evaluation_results)

# Average the metrics across evaluated queries
summary_df = (
    evaluation_df.groupby("K")[
        ["Precision", "Recall", "HitRate"]
    ]
    .mean()
    .reset_index()
)

print("FAISS Retrieval Evaluation Results:")
display(summary_df.round(4))

FAISS Retrieval Evaluation Results:


,K,Precision,Recall,HitRate
0,5,0.4336,0.0296,0.8684
1,10,0.3766,0.0507,0.9342
2,20,0.3062,0.0808,0.9803


In [22]:
# Store comparison results
comparison_metrics = []

# Evaluate the baseline and query-expansion approaches
for approach in ["Baseline", "Query Expansion"]:

    for _, row in valid_queries.iterrows():

        q_id = str(row["q_id"])
        job_title = str(row["jobtitle"])

        # Approach 1: Use the original job title
        if approach == "Baseline":
            query_text = job_title

        # Approach 2: Add general occupational context
        else:
            query_text = (
                f"Occupation: {job_title}. "
                "Relevant professional skills, work tasks, "
                "knowledge, and abilities associated with this role."
            )

        # Encode the selected query
        query_embedding = embedding_model.encode(
            [query_text],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        # Retrieve the top 20 candidate skills
        scores, indices = skill_index.search(query_embedding, 20)

        retrieved_ids = (
            skills_df.iloc[indices[0]]["c_id"]
            .astype(str)
            .tolist()
        )

        relevant_ids = qrels_relevant.get(q_id, set())

        if not relevant_ids:
            continue

        # Calculate metrics at different K values
        for k in [5, 10, 20]:

            top_ids = retrieved_ids[:k]
            relevant_count = len(set(top_ids) & relevant_ids)

            comparison_metrics.append({
                "approach": approach,
                "q_id": q_id,
                "K": k,
                "Precision": relevant_count / k,
                "Recall": relevant_count / len(relevant_ids),
                "HitRate": int(relevant_count > 0)
            })

# Summarize the results
comparison_df = pd.DataFrame(comparison_metrics)

comparison_summary = (
    comparison_df
    .groupby(["approach", "K"])[
        ["Precision", "Recall", "HitRate"]
    ]
    .mean()
    .reset_index()
)

print("Baseline vs Query Expansion:")
display(comparison_summary.round(4))

Baseline vs Query Expansion:


,approach,K,Precision,Recall,HitRate
0,Baseline,5,0.4336,0.0296,0.8684
1,Baseline,10,0.3766,0.0507,0.9342
2,Baseline,20,0.3062,0.0808,0.9803
3,Query Expansion,5,0.3618,0.0246,0.8289
4,Query Expansion,10,0.2934,0.0397,0.9243
5,Query Expansion,20,0.2436,0.0643,0.9737


In [23]:
def retrieve_skills_semantic(job_title, top_k=10):
    """
    Retrieve the most semantically similar ESCO skills
    for a given job title.
    """

    # Convert the job title into an embedding
    query_embedding = embedding_model.encode(
        [job_title],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Make sure top_k does not exceed the index size
    top_k = min(top_k, skill_index.ntotal)

    # Search the FAISS index
    scores, indices = skill_index.search(
        query_embedding,
        top_k
    )

    # Get the matching skill records
    results = skills_df.iloc[indices[0]][
        ["c_id", "esco_uri", "skill_text"]
    ].copy()

    # Add similarity scores
    results["similarity_score"] = scores[0]

    return results.reset_index(drop=True)


# Test the function
test_results = retrieve_skills_semantic(
    "Machine Learning Engineer",
    top_k=10
)

display(test_results.round(4))

,c_id,esco_uri,skill_text,similarity_score
0,dev_cb_sk_1338,http://data.europa.eu/esco/skill/72a74f69-5cf1...,engineering processes,0.4522
1,dev_cb_sk_1006,http://data.europa.eu/esco/skill/3a2d5b45-56e4...,ML machine learning models supervised learning...,0.4254
2,dev_cb_sk_1225,http://data.europa.eu/esco/skill/25f0ea33-b4a2...,data mining,0.3965
3,dev_cb_sk_1380,http://data.europa.eu/esco/skill/89f6560b-2194...,field of computer engineering software design ...,0.3898
4,dev_cb_sk_1058,http://data.europa.eu/esco/skill/5f5e9350-1d13...,e-learning,0.3791
5,dev_cb_sk_804,http://data.europa.eu/esco/skill/0d73a5c0-7f95...,ocean engineering maritime engineering marine ...,0.3742
6,dev_cb_sk_963,http://data.europa.eu/esco/skill/41ec47dd-08b3...,industrial application of software industrial ...,0.3680
7,dev_cb_sk_793,http://data.europa.eu/esco/skill/143769cb-b61e...,business intelligence,0.3541
8,dev_cb_sk_729,http://data.europa.eu/esco/skill/49812c2d-3a36...,develop classification system formulate classi...,0.3486
9,dev_cb_sk_617,http://data.europa.eu/esco/skill/3ec2e4d6-7000...,data warehouse,0.3461


In [24]:
def build_rag_context(job_title, top_k=10):
    """
    Retrieve skills and format them as context for an LLM.
    """

    retrieved_skills = retrieve_skills(
        job_title,
        top_k=top_k
    )

    context = (
        f"Job Title: {job_title}\n\n"
        "Retrieved ESCO Skills:\n"
    )

    for rank, row in retrieved_skills.iterrows():
        context += (
            f"{rank + 1}. {row['skill_text']} "
            f"(Similarity: {row['similarity_score']:.3f})\n"
        )

    return context


# Test the RAG context builder
job_title = "Machine Learning Engineer"

context = build_rag_context(
    job_title,
    top_k=10
)

print(context)

Job Title: Machine Learning Engineer

Retrieved ESCO Skills:
1. ML machine learning models supervised learning unsupervised learning machine learning semi-supervised learning (Similarity: 0.554)
2. e-learning (Similarity: 0.452)
3. machine tools using operating machine tools machine tool operating use machine tools operate machine tools using machine tools machine tool using machine tools operating (Similarity: 0.238)
4. consult on learning methods confer on learning methods advising on learning methods make recommendations on learning methods advise on learning methods advises on learning methods (Similarity: 0.234)
5. monitor liquefied spray cutting machine oversee water jet cutter machine guard water jet cutter machine monitor water jet cutter machine watch over liquefied spray cutting machine watch over water jet cutter machine guard liquefied spray cutting machine tend water jet cutter machine oversee liquefied spray cutting machine (Similarity: 0.169)
6. provide machine proffer o

In [25]:
# Load a pretrained Cross-Encoder reranking model

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross-Encoder reranker loaded successfully!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder reranker loaded successfully!


In [26]:
# Reuse the existing FAISS index (normalized embeddings -> inner product ranks identically to L2)
faiss_index = skill_index

print("FAISS index reused. Total skill vectors:", faiss_index.ntotal)

FAISS index reused. Total skill vectors: 1439


In [27]:
# Number of candidate skills to retrieve before reranking
candidate_k = 50

# Number of final skills to evaluate
final_k = 10

# Store evaluation results for each query
reranked_results = []

# Generate embeddings for all job titles
query_texts = queries_df["jobtitle"].tolist()

query_embeddings = embedding_model.encode(
    query_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

# Retrieve the top 50 candidates for every query
faiss_scores, faiss_indices = faiss_index.search(
    query_embeddings,
    candidate_k
)

# Process each query
for i, (_, row) in enumerate(queries_df.iterrows()):

    query_id = row["q_id"]
    job_title = row["jobtitle"]

    # Get candidate skill text from the FAISS results
    candidate_indices = [
        idx for idx in faiss_indices[i] if idx >= 0
    ]

    candidate_texts = [
    skills_df.iloc[idx]["skill_text"]
    for idx in candidate_indices
]

    candidate_ids = [
        skills_df.iloc[idx]["c_id"]
        for idx in candidate_indices
    ]

    # Create job-title and skill-text pairs for the Cross-Encoder
    pairs = [
        (job_title, skill_text)
        for skill_text in candidate_texts
    ]

    # Score every candidate pair
    rerank_scores = reranker.predict(pairs)

    # Sort candidates by Cross-Encoder score, highest first
    ranked_positions = np.argsort(rerank_scores)[::-1]

    reranked_ids = [
        candidate_ids[pos]
        for pos in ranked_positions[:final_k]
    ]

    # Get the known relevant skills for this query
    relevant = relevant_skills.get(query_id, set())

    # Calculate Precision@10
    hits = sum(
        skill_id in relevant for skill_id in reranked_ids
    )

    precision_at_10 = hits / final_k

    # Calculate Recall@10
    recall_at_10 = (
        hits / len(relevant)
        if relevant else np.nan
    )

    # Calculate reciprocal rank
    reciprocal_rank = 0.0

    for rank, skill_id in enumerate(reranked_ids, start=1):
        if skill_id in relevant:
            reciprocal_rank = 1 / rank
            break

    reranked_results.append({
        "query_id": query_id,
        "jobtitle": job_title,
        "precision_at_10": precision_at_10,
        "recall_at_10": recall_at_10,
        "reciprocal_rank": reciprocal_rank
    })

# Save all per-query results
reranked_df = pd.DataFrame(reranked_results)

# Calculate average reranked metrics
print("Queries evaluated:", len(reranked_df))
print(
    "Reranked Precision@10:",
    round(reranked_df["precision_at_10"].mean(), 4)
)
print(
    "Reranked Recall@10:",
    round(reranked_df["recall_at_10"].mean(), 4)
)
print(
    "Reranked MRR:",
    round(reranked_df["reciprocal_rank"].mean(), 4)
)

display(reranked_df.head())


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Queries evaluated: 304
Reranked Precision@10: 0.4033
Reranked Recall@10: 0.0552
Reranked MRR: 0.7067


,query_id,jobtitle,precision_at_10,recall_at_10,reciprocal_rank
0,dev_qb_jt_1,corporate governance analyst,0.5,0.052083,1.000000
1,dev_qb_jt_2,logistics business analyst,0.7,0.057377,1.000000
2,dev_qb_jt_3,operations planning analyst,0.6,0.051724,1.000000
3,dev_qb_jt_4,real estate data analyst,0.5,0.063291,1.000000
4,dev_qb_jt_5,security analyst ii,0.4,0.037037,0.333333


In [28]:

# Step 2: Inspect the number of relevant skills per query

# Count relevant skills for every validation query
relevant_counts_df = pd.DataFrame([
    {
        "q_id": q_id,
        "relevant_skill_count": len(skill_ids)
    }
    for q_id, skill_ids in qrels_relevant.items()
])

# Display summary statistics
print("Relevant skills per query:")
display(
    relevant_counts_df["relevant_skill_count"]
    .describe()
    .to_frame()
    .round(2)
)

# Display the queries with the largest relevance sets
print("Top 10 queries by number of relevant skills:")
display(
    relevant_counts_df
    .sort_values("relevant_skill_count", ascending=False)
    .head(10)
)

# Check how many queries have unusually large relevance sets
print(
    "Queries with more than 100 relevant skills:",
    (relevant_counts_df["relevant_skill_count"] > 100).sum()
)


Relevant skills per query:


,relevant_skill_count
count,304.00
mean,85.16
std,32.80
min,21.00
25%,59.00
50%,82.00
75%,108.25
max,179.00


Top 10 queries by number of relevant skills:


,q_id,relevant_skill_count
95,dev_qb_jt_185,179
213,dev_qb_jt_291,175
258,dev_qb_jt_58,166
152,dev_qb_jt_236,164
142,dev_qb_jt_227,162
23,dev_qb_jt_12,160
72,dev_qb_jt_164,155
127,dev_qb_jt_213,153
277,dev_qb_jt_75,153
22,dev_qb_jt_119,151


Queries with more than 100 relevant skills: 97


In [29]:

import os
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Enter your Groq API key securely
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

# Initialize the Groq language model
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Test the model before running the full workflow
response = llm.invoke("Explain job skill extraction in one sentence.")
print(response.content)


Job skill extraction is the automated process of identifying and pulling out specific competencies, qualifications, and abilities mentioned in job postings or resumes using natural‑language‑processing techniques.


In [30]:
# Create the prompt for generating job skill explanations
rag_prompt = ChatPromptTemplate.from_template("""
You are an AI assistant specializing in job skill extraction.

Job Title: {job_title}

Retrieved Skills:
{skill_context}

Instructions:
1. Identify the most relevant skills for the given job title.
2. Explain briefly why these skills are relevant.
3. Use only the provided retrieved skills as evidence.
4. Do not invent skills that are not present in the context.
5. Mention that the results are predictions and may require human validation.

Provide a clear, structured explanation.
""")

print("RAG prompt created successfully!")

RAG prompt created successfully!


In [31]:

# Connect the prompt, Groq LLM, and output parser
rag_chain = rag_prompt | llm | StrOutputParser()

print("RAG chain initialized successfully!")


RAG chain initialized successfully!


In [32]:
from typing import TypedDict

# Define the state shared between LangGraph agent nodes
class AgentState(TypedDict):
    job_title: str
    retrieved_candidates: list
    reranked_skills: list
    rag_analysis: str

print("AgentState defined successfully!")

AgentState defined successfully!


In [33]:
def agent_retrieve_skills(state: AgentState):
    """Retrieve candidate skills for the given job title."""

    job_title = state["job_title"]

    # Generate an embedding for the job title
    query_embedding = embedding_model.encode(
        [job_title],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve the top 50 candidate skills from FAISS
    candidate_k = 50
    scores, indices = faiss_index.search(
        query_embedding,
        candidate_k
    )

    # Convert FAISS indices into skill records
    candidates = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        skill = skills_df.iloc[idx]

        candidates.append({
            "c_id": skill["c_id"],
            "skill_text": skill["skill_text"],
            "faiss_score": float(score)
        })

    print(f"[Agent Node 1] Retrieved {len(candidates)} candidate skills.")

    return {"retrieved_candidates": candidates}

In [34]:
def agent_rerank_skills(state: AgentState):
    """Rerank retrieved skills using the Cross-Encoder."""

    job_title = state["job_title"]
    candidates = state["retrieved_candidates"]

    print("[Agent Node 2] Reranking candidate skills...")

    if not candidates:
        return {"reranked_skills": []}

    # Create job-title and skill-text pairs
    pairs = [
        (job_title, skill["skill_text"])
        for skill in candidates
    ]

    # Calculate Cross-Encoder relevance scores
    scores = reranker.predict(pairs)

    # Attach scores to the candidate skills
    reranked = []

    for skill, score in zip(candidates, scores):
        reranked.append({
            **skill,
            "rerank_score": float(score)
        })

    # Sort by relevance score, highest first
    reranked.sort(
        key=lambda skill: skill["rerank_score"],
        reverse=True
    )

    # Keep the top 10 skills
    reranked = reranked[:10]

    print(f"Selected {len(reranked)} skills after reranking.")

    return {"reranked_skills": reranked}

In [35]:

def agent_generate_explanation(state: AgentState):
    """Generate a RAG explanation using the Groq LLM."""

    # Get the job title and reranked skills from the agent state
    job_title = state["job_title"]
    reranked = state["reranked_skills"]

    print("[Agent Node 3] Generating RAG explanation using Groq...")

    # Handle the case where no skills were retrieved
    if not reranked:
        return {
            "rag_analysis": "No relevant skills were retrieved for this job title."
        }

    # Prepare skill information as context for the LLM
    skill_context = "\n".join([
        f"Skill ID: {skill['c_id']}; "
        f"Skill: {skill['skill_text']}; "
        f"Cross-Encoder Score: {skill['rerank_score']:.4f}"
        for skill in reranked
    ])

    # Generate the explanation using the RAG chain
    answer = rag_chain.invoke({
        "job_title": job_title,
        "skill_context": skill_context
    })

    # Return the result to the LangGraph state
    return {"rag_analysis": answer}


print("Explanation agent function defined successfully!")


Explanation agent function defined successfully!


In [36]:
from langgraph.graph import StateGraph, END

# Re-compile the graph structure
workflow = StateGraph(AgentState)
workflow.add_node("retrieve", agent_retrieve_skills)
workflow.add_node("rerank", agent_rerank_skills)
workflow.add_node("generate_rag", agent_generate_explanation)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "rerank")
workflow.add_edge("rerank", "generate_rag")
workflow.add_edge("generate_rag", END)

skill_extraction_agent = workflow.compile()
print("LangGraph Agent Workflow re-compiled successfully!")

# Invoke the agent
initial_state = {
    "job_title": "AI Research Scientist",
    "retrieved_candidates": [],
    "reranked_skills": [],
    "rag_analysis": ""
}

final_output = skill_extraction_agent.invoke(initial_state)
print("\n" + "="*50)
print(f"AGENT EXECUTION RESULTS FOR: {final_output['job_title']}")
print("="*50)
print("\nRAG Analysis Output:\n", final_output.get("rag_analysis"))

LangGraph Agent Workflow re-compiled successfully!
[Agent Node 1] Retrieved 50 candidate skills.
[Agent Node 2] Reranking candidate skills...
Selected 10 skills after reranking.
[Agent Node 3] Generating RAG explanation using Groq...

AGENT EXECUTION RESULTS FOR: AI Research Scientist

RAG Analysis Output:
 **Predicted Relevant Skills for “AI Research Scientist”**  
*(These are predictions based on the retrieved skill set and should be reviewed by a human recruiter or hiring manager.)*  

| Rank | Skill ID | Skill (as retrieved) | Why it is relevant to an AI Research Scientist |
|------|----------|----------------------|-----------------------------------------------|
| 1 | **dev_cb_sk_355** | *principles of artificial intelligence principles of AI* | Core knowledge of AI theory, algorithms, and ethical considerations is the foundation for any research work in this field. |
| 2 | **dev_cb_sk_1057** | *conduct scientific research performing scientific research engage in scientific resea